# Bayesian methods for forecasting: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.statespace.structural import UnobservedComponents

rng = np.random.default_rng(7)

### 1. Normal-normal: prior mu ~ N(m0, tau^2), data with known sigma

In [ ]:
# 1. Normal-normal: prior mu ~ N(m0, tau^2), data with known sigma
m0, tau, sigma, n, ybar = 50, 4.0, 8.0, 9, 56.0
prec = 1 / tau ** 2 + n / sigma ** 2
v = 1 / prec
m = v * (m0 / tau ** 2 + n * ybar / sigma ** 2)
w_data = (n / sigma ** 2) / prec
print("posterior mean:", round(m, 3), "| variance:", round(v, 3), "| 95% interval:", np.round([m - 1.96 * np.sqrt(v), m + 1.96 * np.sqrt(v)], 2))
print("weight on the data:", round(w_data, 3), "| predictive sd:", round(np.sqrt(v + sigma ** 2), 3))

### 2. Conjugate AR(1) with known sigma = 1: prior phi ~ N(0, 0.4^2)

In [ ]:
# 2. Conjugate AR(1) with known sigma = 1: prior phi ~ N(0, 0.4^2)
phi_true, N = 0.7, 40
y = np.zeros(N + 1)
for t in range(1, N + 1):
    y[t] = phi_true * y[t - 1] + rng.standard_normal()
Sxy, Sxx = np.sum(y[1:] * y[:-1]), np.sum(y[:-1] ** 2)
mu0, s0 = 0.0, 0.4
pp = 1 / s0 ** 2 + Sxx
pm = (mu0 / s0 ** 2 + Sxy) / pp
ps = 1 / np.sqrt(pp)
print("\nLS estimate:", round(Sxy / Sxx, 3), "| posterior mean:", round(pm, 3), "| posterior sd:", round(ps, 3), "| true:", phi_true)

### 3. Posterior predictive: draw phi, then simulate the future

In [ ]:
# 3. Posterior predictive: draw phi, then simulate the future
H, S = 12, 4000
phis = np.clip(pm + ps * rng.standard_normal(S), -0.995, 0.995)
paths = np.zeros((S, H)); last = np.full(S, y[-1])
for k in range(H):
    last = phis * last + rng.standard_normal(S)
    paths[:, k] = last
lo, md, hi = np.percentile(paths, [2.5, 50, 97.5], axis=0)
plug = y[-1] * (Sxy / Sxx) ** np.arange(1, H + 1)
plug_sd = np.sqrt(np.cumsum((Sxy / Sxx) ** (2 * np.arange(H))))
print("h=12: Bayes 95% band", np.round([lo[-1], hi[-1]], 2), "(width", round(hi[-1] - lo[-1], 2), ") | plug-in band width", round(2 * 1.96 * plug_sd[-1], 2))

### 4. A strong wrong prior is overcome by data

In [ ]:
# 4. A strong wrong prior is overcome by data
for N2 in (10, 40, 200):
    yy = np.zeros(N2 + 1)
    for t in range(1, N2 + 1):
        yy[t] = phi_true * yy[t - 1] + rng.standard_normal()
    a, b = np.sum(yy[1:] * yy[:-1]), np.sum(yy[:-1] ** 2)
    s_p = 0.1
    post = (0 / s_p ** 2 + a) / (1 / s_p ** 2 + b)
    print(f"prior N(0, 0.1^2), n={N2:3d}: posterior mean {post:.3f} (LS {a / b:.3f}, true 0.7)")

### 5. Local level model and the Kalman filter on the Nile data

In [ ]:
# 5. Local level model and the Kalman filter on the Nile data
nile = sm.datasets.nile.load_pandas().data["volume"].values.astype(float)
res = UnobservedComponents(nile, level="llevel").fit(disp=False)
print("\nNile local level: sigma2_irregular =", round(res.params[0], 0), "| sigma2_level =", round(res.params[1], 0))
q = res.params[1] / res.params[0]
print("signal-to-noise ratio q:", round(q, 3))
P = res.filter_results.predicted_state_cov[0, 0, -1]
K = res.filter_results.kalman_gain[0, 0, -1]
print("final Kalman gain:", round(K, 3), "| predicted state variance:", round(P, 0))
fc = res.get_forecast(5)
print("5-step forecast mean:", np.round(fc.predicted_mean, 0), "(flat for a local level model)")

### 6. One hand-computed Kalman step: m=30, P=9, Q=2, R=4, y=34

In [ ]:
# 6. One hand-computed Kalman step: m=30, P=9, Q=2, R=4, y=34
m1, P1, Q, R, obs = 30, 9, 2, 4, 34
Pp = P1 + Q; K = Pp / (Pp + R)
print("\nKalman gain:", round(K, 4), "| new mean:", round(m1 + K * (obs - m1), 3), "| new variance:", round((1 - K) * Pp, 3), "| forecast variance:", round((1 - K) * Pp + Q + R, 3))

## Expected output

Running all cells above should print:

```
posterior mean: 54.154 | variance: 4.923 | 95% interval: [49.8 58.5]
weight on the data: 0.692 | predictive sd: 8.302

LS estimate: 0.883 | posterior mean: 0.842 | posterior sd: 0.086 | true: 0.7
h=12: Bayes 95% band [-4.18  3.81] (width 7.99 ) | plug-in band width 8.14
prior N(0, 0.1^2), n= 10: posterior mean 0.078 (LS 0.600, true 0.7)
prior N(0, 0.1^2), n= 40: posterior mean 0.171 (LS 0.500, true 0.7)
prior N(0, 0.1^2), n=200: posterior mean 0.567 (LS 0.710, true 0.7)

Nile local level: sigma2_irregular = 15078.0 | sigma2_level = 1479.0
signal-to-noise ratio q: 0.098
final Kalman gain: 0.268 | predicted state variance: 5519.0
5-step forecast mean: [798. 798. 798. 798. 798.] (flat for a local level model)

Kalman gain: 0.7333 | new mean: 32.933 | new variance: 2.933 | forecast variance: 8.933
```

## Practice exercises

Try these in the cells below.

1. Use a weak prior (large prior sd) and compare the posterior mean of φ with the least squares estimate.
2. Use a strong wrong prior (mean 0.2, sd 0.05) with a short series. How does the posterior change when n grows?
3. Compare the width of the forecast bands for n = 20 and n = 200.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
